In [ ]:
import torch; print(f"🔥 CUDA Disponible: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Ninguna'}")

In [1]:
# ==============================================================================
# 1. LIBRERÍAS Y CONFIGURACIÓN DEL ENTORNO (PEP 8)
# ==============================================================================
import os
import sys
import time
from functools import reduce

# PySpark Core y Delta
import pyspark
import delta
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import LongType
from pyspark.sql.types import *

# Componentes de PySpark ML
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator



# Ecosistema Extendido (XGBoost y Deep Learning)
from xgboost.spark import SparkXGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
# Inicialización de la Spark Session Optimizada para Contenedores de ML (Java 17)


In [2]:
#spark.stop()

In [ ]:
spark = (SparkSession.builder
    .appName("Portfolio-Risk-Engine")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.3.2")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "20g")
    .config("spark.executor.memory", "20g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
    .getOrCreate()
)

# ==============================================================================
# 2. INGESTA DE DATOS Y PREPARACIÓN DE CARACTERÍSTICAS
# ==============================================================================
# Rutas internas del contenedor montado en el HDD
path_clientes = "/warehouse/pool/clientes"
path_historial = "/warehouse/pool/historial"

df_clientes_raw = spark.read.format("delta").load(path_clientes)
df_historial_raw = spark.read.format("delta").load(path_historial)

# ==============================================================================
# RECONSTRUCCIÓN DE CTACLIENTE (A PRUEBA DE SOBREPARTICIÓN)
# ==============================================================================

# 1. Fuerza bruta: Cortamos los primeros 10 caracteres (YYYY-MM-DD)
ventana_unicidad = Window.partitionBy(F.substring("FECHA_ALTA", 1, 10)) \
                         .orderBy("id_cliente", "fecha_envio", "batch_id")

df_clientes_raw = (
    df_clientes_raw.withColumn( # A: Extraemos la fecha en 6 dígitos (YYMMDD) forzando el formato a la fecha cortada
        "fecha_6d", F.date_format(F.to_date(F.substring("FECHA_ALTA", 1, 10)), "yyyyMMdd")
    ).withColumn( # B: Generamos el número de fila (Spark ahora procesará por bloques diarios)
        "secuencia_diaria",  F.row_number().over(ventana_unicidad)
    ).withColumn( # C: Rellenamos con ceros a la izquierda (7 dígitos)
    "secuencia_7d", F.lpad(F.col("secuencia_diaria").cast("string"), 7, "0")
    ).withColumn(# D: Concatenamos y convertimos a Long (14 dígitos perfectos)
        "CTACLIENTE", F.concat( F.lit("1"),  F.col("fecha_6d"),  F.col("secuencia_7d") ).cast(LongType())
    ).drop("fecha_6d", "secuencia_diaria", "secuencia_7d")
)

# Agregaciones del historial usando columnas reales del esquema
df_features_hist = df_historial_raw.groupBy("id_cliente", "fecha_envio", "batch_id").agg(
    F.avg("pago_realizado").alias("avg_gasto"),
    F.stddev("pago_realizado").alias("std_gasto"),
    F.count("mes_vida").alias("frecuencia_uso"),
    F.max("pago_realizado").alias("max_gasto"),
    F.avg("pago_fijo").alias("pago_fijo"),
    
    # Target 1: Crédito (¿Tuvo mora alguna vez? 0 o 1)
    F.max("mora").alias("mora"),    
    # Target 2: Liquidez (Binarización directa en la agregación)
    F.max("Ind_Pago_Minimo").alias("Ind_Pago_Minimo"),    
    # Target 3: Estatus (El valor de 'estatus' correspondiente a la última 'fecha_corte' o 'mes_vida')
    F.max_by("estatus", "fecha_corte").alias("estatus")
    
).fillna(0, subset=["std_gasto", "avg_gasto"])

# Dataset unificado para modelado
df_maestro = df_clientes_raw.join(df_features_hist, on=["id_cliente", "fecha_envio", "batch_id"], how="inner")

#df_maestro.show(5)





# ==============================================================================
# 3. MOTOR DE EVALUACIÓN ANALÍTICA (KPIs Y MATRIZ DE CONFUSIÓN)
# ==============================================================================
def evaluar_rendimiento_modelo(df_pred, col_target, col_pred, col_prob, es_multiclase=False):
    """
    Calcula e imprime los KPIs de rendimiento del modelo y su matriz de confusión.
    """
    print(f"\n{'='*60}")
    print(f"📊 REPORTE DE EVALUACIÓN: TARGET [{col_target}] -> MODELO [{col_pred}]")
    print(f"{'='*60}")
    
    # 1. Métricas basadas en clases (Precisión, Recall, F1)
    eval_multi = MulticlassClassificationEvaluator(labelCol=col_target, predictionCol=col_pred)
    precision = eval_multi.evaluate(df_pred, {eval_multi.metricName: "weightedPrecision"})
    recall = eval_multi.evaluate(df_pred, {eval_multi.metricName: "weightedRecall"})
    f1_score = eval_multi.evaluate(df_pred, {eval_multi.metricName: "f1"})
    
    print(f"✅ Precisión Ponderada: {precision:.4f}")
    print(f"✅ Recall Ponderada:    {recall:.4f}")
    print(f"✅ F1-Score Ponderado:  {f1_score:.4f}")
    
    # 2. Métricas de Discriminación (Solo para Clasificación Binaria)
    if not es_multiclase and col_prob:
        eval_binary = BinaryClassificationEvaluator(labelCol=col_target, rawPredictionCol=col_prob, metricName="areaUnderROC")
        roc_auc = eval_binary.evaluate(df_pred)
        print(f"✅ Área Bajo la Curva ROC (AUC): {roc_auc:.4f}")
        
    # 3. Matriz de Confusión / Distribución de Predicciones Nativas
    print("\n🔹 Matriz de Confusión (Frecuencias Cruzadas):")
    df_pred.groupBy(col_target, col_pred).count().orderBy(col_target, col_pred).show()



# ==============================================================================
# 4. FÁBRICA DE PIPELINES DE MACHINE LEARNING (CON CUARENTENA DE NULOS)
# ==============================================================================
def fabrica_modelos_ml(df, nombre_config, config):
    print(f"\n⚙️ Configurando pipeline para {nombre_config}...")
    
    target = config["target"]
    features = config["features"]
    tipo_modelo = config["type"]
    es_multiclase = config.get("multiclass", False)
    
    # Aislamiento dinámico de nulos a una tabla de cuarentena
    columnas_requeridas = features + [target]
    condicion_nulos = reduce(lambda a, b: a | b, (F.col(c).isNull() for c in columnas_requeridas))
    
    df_cuarentena = df.filter(condicion_nulos)
    df_entrenamiento = df.filter(~condicion_nulos)
    
    if df_cuarentena.count() > 0:
        print(f"⚠️ Alerta: {df_cuarentena.count()} filas enviadas a la tabla de nulos.")
        # Aquí se podría escribir df_cuarentena a una ruta delta de auditoría
        
    # Construcción de las etapas del Pipeline de Spark
    stages = []
    
    # Si el target es string (ej. estatus para clasificación multiclase), lo indexamos
    if es_multiclase and df_entrenamiento.schema[target].dataType == StringType():
        target_indexed = f"{target}_indexed"
        indexer = StringIndexer(inputCol=target, outputCol=target_indexed, handleInvalid="skip")
        stages.append(indexer)
        target_final = target_indexed
    else:
        target_final = target
        
    assembler = VectorAssembler(inputCols=features, outputCol="features_raw", handleInvalid="skip")
    scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withStd=True, withMean=True)
    stages.extend([assembler, scaler])
    
    # Selección dinámica del Algoritmo
    if tipo_modelo == "random_forest":
        algo = RandomForestClassifier(featuresCol="features_scaled", 
                                      labelCol=target_final, 
                                      predictionCol=f"pred_{nombre_config}", 
                                      probabilityCol=f"prob_{nombre_config}", 
                                      seed=42)
    elif tipo_modelo == "gbt":
        algo = GBTClassifier(featuresCol="features_scaled", 
                             labelCol=target_final, 
                             predictionCol=f"pred_{nombre_config}", 
                             maxIter=30, 
                             seed=42)
    elif tipo_modelo == "xgboost":
        algo = SparkXGBClassifier(features_col="features_scaled", 
                                  label_col=target_final, 
                                  prediction_col=f"pred_{nombre_config}", 
                                  probability_col=f"prob_{nombre_config}", 
                                  device="cuda", # <--- ¡TURBO ACTIVADO!                                  
#                                  num_workers=2, 
                                  random_state=42)
    stages.append(algo)
    
    # Entrenamiento del Pipeline
    pipeline = Pipeline(stages=stages)
    model = pipeline.fit(df_entrenamiento)
    predictions = model.transform(df_entrenamiento)
    
    # Evaluación
    prob_col = f"prob_{nombre_config}" if tipo_modelo != "gbt" else None
    evaluar_rendimiento_modelo(predictions, target_final, f"pred_{nombre_config}", prob_col, es_multiclase)
    
    return model, predictions







# DICCIONARIO DE CONFIGURACIÓN DE EXPERIMENTOS
configuracion_experimentos = {
    "Riesgo_Credito_GBT": {
        "type": "gbt",
        "target": "mora", # Clasificación Binaria
        "features": ["SCO_ACT", "SAL_TOTDEU", "UTIL_TC", "NUM_CREDACTI", "plazo_meses"],
        "multiclass": False
    },
    "Riesgo_Liquidez_XGB": {
        "type": "xgboost",
        "target": "Ind_Pago_Minimo", # Clasificación Binaria
        "features": ["ingreso_mensual", "gasto_mensual", "capacidad_ahorro", "DTI", "pago_fijo"],
        "multiclass": False
    },
    "Estatus_Cliente_RF": {
        "type": "random_forest",
        "target": "estatus", # Clasificación Multiclase (Estatus: Activo, Cerrado, Mora)
        "features": ["edad", "ingreso_mensual", "avg_gasto", "frecuencia_uso", "ANT_LAB_MES"],
        "multiclass": True
    }
}

# Ejecución secuencial del stack de Machine Learning
modelos_entrenados = {}
predicciones_ml = {}

for nombre, config in configuracion_experimentos.items():
    modelo, df_pred = fabrica_modelos_ml(df_maestro, nombre, config)
    modelos_entrenados[nombre] = modelo
    predicciones_ml[nombre] = df_pred



# ==============================================================================
# 5. BLOQUE DE DEEP LEARNING (PYTORCH) CON PASO DE DATOS VÍA ARROW
# ==============================================================================
print("\n🔥 Iniciando entrenamiento del modelo de Deep Learning (PyTorch)...")

# Variables para la Red Neuronal
features_dl = ["SCO_ACT", "SAL_TOTDEU", "UTIL_TC", "DTI", "capacidad_ahorro"]
target_dl = "mora"

# Filtro de nulos y conversión ultrarrápida a Pandas usando Apache Arrow en memoria
df_dl_clean = df_maestro.dropna(subset=features_dl + [target_dl])
pdf_dl = df_dl_clean.select(features_dl + [target_dl]).toPandas()


# 1. Detectar si la GPU de NVIDIA está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Entrenando Red Neuronal usando: {device.type.upper()}")

## Conversión a tensores de PyTorch
#X_tensor = torch.tensor(pdf_dl[features_dl].values, dtype=torch.float32)
#y_tensor = torch.tensor(pdf_dl[target_dl].values, dtype=torch.float32).unsqueeze(1)

# 2. Enviar los tensores directamente a la GPU
X_tensor = torch.tensor(pdf_dl[features_dl].values, dtype=torch.float32).to(device)
y_tensor = torch.tensor(pdf_dl[target_dl].values, dtype=torch.float32).unsqueeze(1).to(device)



# Normalización manual Z-score en PyTorch (equivalente a StandardScaler)
X_mean = X_tensor.mean(dim=0)
X_std = X_tensor.std(dim=0) + 1e-8
X_scaled = (X_tensor - X_mean) / X_std

# Preparación del DataLoader
dataset = TensorDataset(X_scaled, y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Definición de la Arquitectura de la Red Neuronal Feed-Forward (Multicapa)
class RedRiesgoFinanciero(nn.Module):
    def __init__(self, input_dim):
        super(RedRiesgoFinanciero, self).__init__()
        self.net = nn.Sequential( nn.Linear(input_dim, 16), nn.ReLU(), nn.Dropout(0.2), nn.Linear(16, 8),
                                  nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid() # Salida de probabilidad para clasificación binaria
        )
        
    def forward(self, x):
        return self.net(x)

# Inicialización del modelo, optimizador y función de pérdida
modelo_dl = RedRiesgoFinanciero(input_dim=len(features_dl)).to(device)
#modelo_dl = RedRiesgoFinanciero(input_dim=len(features_dl))
criterion = nn.BCELoss()
optimizer = optim.Adam(modelo_dl.parameters(), lr=0.005)


# Bucle de Entrenamiento
modelo_dl.train()
for epoch in range(10): # Ajustado a 10 épocas para el portafolio
    loss_acumulada = 0.0
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()
        predicciones = modelo_dl(X_batch)
        loss = criterion(predicciones, y_batch)
        loss.backward()
        optimizer.step()
        loss_acumulada += loss.item() * X_batch.size(0)
    loss_epoch = loss_acumulada / len(dataloader.dataset)
    print(f"  Época {epoch+1}/10 - Loss Promedio: {loss_epoch:.4f}")

# Evaluación del Modelo de Deep Learning
modelo_dl.eval()
with torch.no_grad():
    #y_probs = modelo_dl(X_scaled).numpy()
    y_probs = modelo_dl(X_scaled).cpu().numpy() 
    y_preds = (y_probs >= 0.5).astype(int)
    #y_true = y_tensor.numpy().astype(int)
    y_true = y_tensor.cpu().numpy().astype(int)



# 1. Cambiamos TODO a DoubleType (Spark es súper estricto)
esquema_salida_dl = StructType([
    StructField("target_true", DoubleType(), True), 
    StructField("pred_dl", DoubleType(), True),     
    StructField("prob_dl", DoubleType(), True)
])

# 2. Forzamos la conversión a flotante (float) en Python
datos_eval_dl = [(float(t), float(p), float(pr)) for t, p, pr in zip(y_true, y_preds, y_probs)]

df_eval_dl = spark.createDataFrame(datos_eval_dl, schema=esquema_salida_dl)

# Imprimir KPIs de la Red Neuronal
evaluar_rendimiento_modelo(df_eval_dl, "target_true", "pred_dl", "prob_dl", es_multiclase=False)


# ==============================================================================
# 6. CONSOLIDACIÓN FINAL (COBERTURA DEL 100% DEL PADRÓN)
# ==============================================================================
print("\n💾 Consolidando predicciones y generando Score Integral...")

extraer_prob_rf = F.udf(lambda v: float(v[1]), DoubleType())
extraer_prob_xgb = F.udf(lambda v: float(v[1]), DoubleType())

# df_c = predicciones_ml["Riesgo_Credito_GBT"].select(    "CTACLIENTE", 
#     extraer_prob_rf(F.col("prob_Riesgo_Credito_GBT")).alias("p_credito")
# )
# Extracción para GBT (usando el nombre por defecto 'probability')
df_c = predicciones_ml["Riesgo_Credito_GBT"].select( "CTACLIENTE", 
    extraer_prob_rf(F.col("probability")).alias("p_credito") # <--- FIX AQUÍ
)

df_l = predicciones_ml["Riesgo_Liquidez_XGB"].select(    "CTACLIENTE", 
    extraer_prob_xgb(F.col("prob_Riesgo_Liquidez_XGB")).alias("p_liquidez")
)


# 1. EL SECRETO: Tomamos el padrón COMPLETO como base
df_base = df_maestro.select("CTACLIENTE").distinct()

# 2. LEFT JOIN: Pegamos los scores. Si el cliente no pasó por el modelo, se queda en NULL
df_final_scores = df_base \
    .join(df_c, on="CTACLIENTE", how="left") \
    .join(df_l, on="CTACLIENTE", how="left")

# 3. CÁLCULO: En Spark, (Numero * 0.6) + (NULL * 0.4) da automáticamente NULL
df_final_scores = df_final_scores.withColumn( "score_final_riesgo",
    (F.col("p_credito") * 0.6) + (F.col("p_liquidez") * 0.4)
)

# 4. ETIQUETADO DE ENRUTAMIENTO (Para Superset y el pipeline secundario)
df_final_scores = df_final_scores.withColumn(
    "estado_evaluacion",
    F.when(F.col("score_final_riesgo").isNotNull(), F.lit("Score Integral Completo"))
     .otherwise(F.lit("Datos Insuficientes - Requiere Modelo Reducido"))
)

# Persistencia física en el Data Lake
ruta_maestra_salida = "/warehouse/pool/master_ml_ready"
df_final_scores.write.format("delta").mode("overwrite").save(ruta_maestra_salida)

print(f"🚀 ¡Proceso finalizado! Tabla maestra (con {df_final_scores.count()} clientes) disponible en: {ruta_maestra_salida}")



:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6709d824-a4b6-4142-adda-24769d29afc8;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 132ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   


⚙️ Configurando pipeline para Riesgo_Credito_GBT...


26/07/12 21:08:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                


📊 REPORTE DE EVALUACIÓN: TARGET [mora] -> MODELO [pred_Riesgo_Credito_GBT]


...

[Stage 927:==================================================>    (22 + 2) / 24]


📊 REPORTE DE EVALUACIÓN: TARGET [estatus_indexed] -> MODELO [pred_Estatus_Cliente_RF]


✅ Precisión Ponderada: 0.5037
✅ Recall Ponderada:    0.5597
✅ F1-Score Ponderado:  0.5160

🔹 Matriz de Confusión (Frecuencias Cruzadas):


+---------------+-----------------------+-------+
|estatus_indexed|pred_Estatus_Cliente_RF|  count|
+---------------+-----------------------+-------+
|            0.0|                    0.0|2226383|
|            0.0|                    1.0| 580816|
|            1.0|                    0.0|1287842|
|            1.0|                    1.0| 838378|
|            2.0|                    0.0| 456756|
|            2.0|                    1.0|  85936|
+---------------+-----------------------+-------+


🔥 Iniciando entrenamiento del modelo de Deep Learning (PyTorch)...


🔥 Entrenando Red Neuronal usando: CUDA


In [4]:
print(df_maestro.count())
print(df_final_scores.count())


5479665


[Stage 1173:>                                                       (0 + 8) / 9]

5479665


In [ ]:
spark.stop()